In [3]:
from saxonche import PySaxonProcessor

# Create the main Saxon processor using context manager
# license=False = open-source Saxon-HE edition
with PySaxonProcessor(license=False) as proc:
    
    # Create specialized processors as needed:
    xslt_proc = proc.new_xslt30_processor()  # For XSLT 3.0 transformations
    xquery_proc = proc.new_xquery_processor()  # For XQuery queries
    xpath_proc = proc.new_xpath_processor()    # For XPath expressions
    

In [7]:
proc = PySaxonProcessor(license=False)

# Create a simple XML document in memory
xml_text = """<?xml version="1.0"?>
<library>
    <book id="1">
        <title>Introduction to XML</title>
        <author>John Smith</author>
        <year>2020</year>
    </book>
    <book id="2">
        <title>XQuery Basics</title>
        <author>Jane Doe</author>
        <year>2021</year>
    </book>
    <book id="3">
        <title>XSLT Guide</title>
        <author>Bob Johnson</author>
        <year>2022</year>
    </book>
</library>"""

# Parse it into a Saxon document object
doc = proc.parse_xml(xml_text=xml_text)

# Confirm it worked
print("✓ XML document created in memory")
print(f"Document type: {doc.node_kind_str}")
print(f"Number of children: {doc.size}")


✓ XML document created in memory
Document type: document
Number of children: 1


In [8]:
doc

<library>
   <book id="1">
      <title>Introduction to XML</title>
      <author>John Smith</author>
      <year>2020</year>
   </book>
   <book id="2">
      <title>XQuery Basics</title>
      <author>Jane Doe</author>
      <year>2021</year>
   </book>
   <book id="3">
      <title>XSLT Guide</title>
      <author>Bob Johnson</author>
      <year>2022</year>
   </book>
</library>

In [9]:
# Create an XPath processor
xpath_proc = proc.new_xpath_processor()

# Tell it which document to search
xpath_proc.set_context(xdm_item=doc)

# Run your XPath query
result = xpath_proc.evaluate("//author")

# See the results
print("Authors found:")
print(result)

Authors found:
<author>John Smith</author>

<author>Jane Doe</author>

<author>Bob Johnson</author>


# Understanding Saxon's XDM Data Types

## What is XDM?

**XDM** stands for **XQuery/XPath Data Model**. It's the W3C standard way of representing XML data in memory for processing with XPath, XQuery, and XSLT.

When Saxon parses XML, it converts the text into an **XDM tree structure** - a rich, hierarchical representation that preserves:
- Parent-child relationships
- Document order
- Node types (element, attribute, text, comment, etc.)
- Type information from XML Schema
- Namespaces

## Why Saxon Needs Its Own Types

Python's native types (strings, lists, dicts) can't represent XML's hierarchical structure effectively:
```python
# This Python dict loses important XML information:
{
    "book": {
        "title": "Learning XML",
        "author": "Jane Doe"
    }
}
# ❌ Lost: order, attributes, mixed content, namespaces, types

# Saxon's XDM preserves everything:
doc = proc.parse_xml(xml_text="""
<book id="1" category="programming">
    <title>Learning XML</title>
    <author>Jane Doe</author>
</book>""")
# ✓ Keeps: hierarchy, @id attribute, order, all metadata
```

## The Saxon Type Hierarchy

Saxon provides several interconnected types:
```
PyXdmValue                    ← A sequence of 0+ items
│
├── PyXdmItem                 ← A single item
    │
    ├── PyXdmNode             ← XML nodes (elements, text, attributes)
    │   ├── Document node
    │   ├── Element node
    │   ├── Attribute node
    │   └── Text node
    │
    ├── PyXdmAtomicValue      ← Simple values (string, integer, boolean)
    │
    ├── PyXdmFunctionItem     ← Functions (for advanced use)
    ├── PyXdmArray            ← Arrays (XDM 3.1 feature)
    └── PyXdmMap              ← Maps (XDM 3.1 feature)
```

## Working with XDM Types

### Creating XDM from Python
```python
# Parse XML string → XDM document tree
doc = proc.parse_xml(xml_text="<book><title>XML Guide</title></book>")
# Type: PyXdmNode (specifically, a document node)

# Convert Python values → XDM atomic values
saxon_string = proc.make_string_value("hello")    # PyXdmAtomicValue
saxon_int = proc.make_integer_value(42)           # PyXdmAtomicValue
saxon_bool = proc.make_boolean_value(True)        # PyXdmAtomicValue
```

### Query Results Return XDM
```python
xpath_proc = proc.new_xpath_processor()
xpath_proc.set_context(xdm_item=doc)

# XPath returns PyXdmValue (a sequence)
result = xpath_proc.evaluate("//title")
print(type(result))  # <class 'saxonche.PyXdmValue'>
print(result.size)   # Number of items in sequence: 1
```

### Converting XDM Back to Python
```python
# Get individual items from sequence
result = xpath_proc.evaluate("//author/text()")

for i in range(result.size):
    item = result.item_at(i)              # PyXdmAtomicValue
    python_string = item.string_value     # Back to Python str
    print(python_string)
```

### Understanding Sequences

**XDM sequences** are fundamental - everything in XPath/XQuery operates on sequences:
```python
# Single item = sequence of length 1
single = xpath_proc.evaluate("//book[1]")
print(single.size)  # 1

# Multiple items = sequence of length N
multiple = xpath_proc.evaluate("//book")
print(multiple.size)  # 3

# Empty result = sequence of length 0
empty = xpath_proc.evaluate("//nonexistent")
print(empty.size)  # 0

# Note: XDM sequences are FLAT - no nested sequences
```

### Node vs Atomic Value
```python
# Returns PyXdmNode objects (the <author> elements with tags)
elements = xpath_proc.evaluate("//author")
item = elements.item_at(0)
print(type(item))           # PyXdmNode
print(item.node_kind_str)   # "element"
print(item.name)            # "author"

# Returns PyXdmAtomicValue objects (just the text content)
text = xpath_proc.evaluate("//author/text()")
item = text.item_at(0)
print(type(item))           # PyXdmAtomicValue
print(item.string_value)    # "John Smith"
```

## Common Patterns
```python
# Pattern 1: Check if results exist
result = xpath_proc.evaluate("//book[@id='5']")
if result.size > 0:
    print("Book found!")
    
# Pattern 2: Get single value
result = xpath_proc.evaluate("count(//book)")
count = result.item_at(0).integer_value  # Convert to Python int

# Pattern 3: Iterate through multiple results
results = xpath_proc.evaluate("//title/text()")
for i in range(results.size):
    title = results.item_at(i).string_value
    print(f"Title: {title}")
    
# Pattern 4: Check node type
node = doc.children[0]  # Get root element
if node.node_kind_str == "element":
    print(f"Element name: {node.name}")
```

---

# Analogy: Saxon XDM is Like Pandas DataFrames

Now that you understand XDM, here's a helpful comparison to another domain-specific data structure you might know:

## Saxon XDM ≈ Pandas DataFrame

| Library | Domain-Specific Type | Purpose |
|---------|---------------------|---------|
| **Pandas** | `DataFrame`, `Series` | Optimized for tabular/statistical data |
| **Saxon** | `PyXdmValue`, `PyXdmNode` | Optimized for hierarchical XML data |

## Why Both Exist

Both libraries create specialized types because **Python's native types aren't sufficient**:
```python
# Pandas: Python lists aren't efficient for data analysis
data = [[1, 2], [3, 4], [5, 6]]        # Python list of lists
df = pd.DataFrame(data)                 # Convert to DataFrame
result = df.mean()                      # Efficient operations

# Saxon: Python strings/dicts don't preserve XML structure
xml_string = "<book><title>XML</title></book>"   # Python string
doc = proc.parse_xml(xml_text=xml_string)        # Convert to XDM tree
result = xpath_proc.evaluate("//title")          # Tree navigation
```

## The Conversion Pattern

Both require conversion between Python types and specialized types:
```python
# PANDAS
# Python → Pandas
python_list = [1, 2, 3, 4, 5]
series = pd.Series(python_list)

# Pandas → Python
value = series.mean()                # Pandas operation
python_float = float(value)          # Back to Python


# SAXON
# Python → Saxon
python_string = "Hello"
saxon_value = proc.make_string_value(python_string)

# Saxon → Python
result = xpath_proc.evaluate("//title/text()")
python_string = result.item_at(0).string_value
```

## Similar Operations

Both provide domain-specific query languages:
```python
# Pandas: Use DataFrame operations/SQL-like syntax
df[df['price'] < 30]                   # Filter rows
df['price'].mean()                     # Aggregate
df.groupby('category').sum()          # Group by

# Saxon: Use XPath/XQuery
xpath_proc.evaluate("//book[price < 30]")        # Filter nodes
xpath_proc.evaluate("avg(//book/price)")         # Aggregate
xpath_proc.evaluate("//book/@category/distinct-values()")  # Distinct
```

## Memory Representations

Both maintain rich in-memory structures:

| Pandas | Saxon |
|--------|-------|
| DataFrame = table with rows/columns | XDM Document = tree with nodes |
| Preserves column types, index | Preserves hierarchy, types, order |
| Optimized for vectorized operations | Optimized for tree traversal |
| Series = 1-dimensional data | PyXdmValue = sequence of items |

## Bottom Line

**Pandas DataFrame** : tabular data :: **Saxon XDM** : hierarchical XML data

Both are:
- ✅ Domain-specific abstractions
- ✅ More powerful than Python's native types for their domain
- ✅ Require type conversion to/from Python
- ✅ Provide specialized query methods
- ✅ Worth the learning curve for their respective use cases

<document artifact_identifier="saxon-python-guide" title="Saxon Python Practical Guide" type="text/markdown">
# Practical Guide: Reading XML and Running Queries/Transformations

## Setup (Run Once)
```python
from saxonche import PySaxonProcessor
import os

# Create Saxon processor
proc = PySaxonProcessor(license=False)
proc.set_cwd(os.getcwd())
```

---

## Step 1: Load Your XML Document

### Option A: From a File
```python
# Read XML from disk
doc = proc.parse_xml(xml_file_name="mydata.xml")
```

### Option B: From a String
```python
# Read XML from variable
xml_text = """<?xml version="1.0"?>
<library>
    <book><title>XML Guide</title><author>John</author></book>
    <book><title>XQuery Basics</title><author>Jane</author></book>
</library>"""

doc = proc.parse_xml(xml_text=xml_text)
```

Now `doc` contains your XML in memory, ready to query.

---

## Step 2: Run XPath

**XPath** = Simple path expressions to find nodes
```python
# Create XPath processor
xpath_proc = proc.new_xpath_processor()
xpath_proc.set_context(xdm_item=doc)

# Run XPath queries
titles = xpath_proc.evaluate("//title/text()")
authors = xpath_proc.evaluate("//author/text()")
first_book = xpath_proc.evaluate("//book[1]/title/text()")

# Get results
for i in range(titles.size):
    print(titles.item_at(i).string_value)
```

**Common XPath patterns:**
- `//title` - All title elements
- `//title/text()` - Just the text inside titles
- `//book[1]` - First book
- `//book[@id='5']` - Book with id attribute = 5
- `count(//book)` - Count books

---

## Step 3: Run XQuery

**XQuery** = Full query language with loops, conditions, etc.

### Option A: Query from String
```python
# Create XQuery processor
xquery_proc = proc.new_xquery_processor()
xquery_proc.set_context(xdm_item=doc)

# Run XQuery
query = """
for $book in //book
where $book/author = 'Jane'
return $book/title/text()
"""

result = xquery_proc.run_query_to_string(query_text=query)
print(result)
```

### Option B: Query from File
```python
# Save query to file: myquery.xq
# for $b in //book
# return <result>{$b/title}</result>

xquery_proc = proc.new_xquery_processor()
xquery_proc.set_context(xdm_item=doc)
result = xquery_proc.run_query_to_string(query_file_name="myquery.xq")
print(result)
```

---

## Step 4: Run XSLT

**XSLT** = Transform XML into different format (HTML, text, different XML)

### Option A: XSLT from String
```python
# Create XSLT processor
xslt_proc = proc.new_xslt30_processor()

# XSLT stylesheet as string
xslt_code = """<?xml version="1.0"?>
<xsl:stylesheet version="3.0" xmlns:xsl="http://www.w3.org/1999/XSL/Transform">
    <xsl:template match="/">
        <html>
            <body>
                <h1>Books</h1>
                <ul>
                    <xsl:for-each select="//book">
                        <li><xsl:value-of select="title"/></li>
                    </xsl:for-each>
                </ul>
            </body>
        </html>
    </xsl:template>
</xsl:stylesheet>"""

# Compile and run
executable = xslt_proc.compile_stylesheet(stylesheet_text=xslt_code)
result = executable.transform_to_string(xdm_node=doc)
print(result)
```

### Option B: XSLT from File
```python
# Save stylesheet to file: transform.xsl
xslt_proc = proc.new_xslt30_processor()

# Compile stylesheet
executable = xslt_proc.compile_stylesheet(stylesheet_file="transform.xsl")

# Transform to string
result = executable.transform_to_string(xdm_node=doc)
print(result)

# Or save to file
executable.transform_to_file(xdm_node=doc, output_file="output.html")
```

---

## Complete Working Example
```python
from saxonche import PySaxonProcessor

# Setup
proc = PySaxonProcessor(license=False)

# Load XML
xml = """<?xml version="1.0"?>
<books>
    <book id="1"><title>XML Guide</title><price>29.99</price></book>
    <book id="2"><title>XQuery</title><price>39.99</price></book>
</books>"""
doc = proc.parse_xml(xml_text=xml)

# XPath: Get all titles
xpath_proc = proc.new_xpath_processor()
xpath_proc.set_context(xdm_item=doc)
titles = xpath_proc.evaluate("//title/text()")
print("XPath found:", titles)

# XQuery: Get cheap books
xquery_proc = proc.new_xquery_processor()
xquery_proc.set_context(xdm_item=doc)
query = "for $b in //book where $b/price < 35 return $b/title/text()"
result = xquery_proc.run_query_to_string(query_text=query)
print("XQuery found:", result)

# XSLT: Transform to HTML
xslt_proc = proc.new_xslt30_processor()
xslt = """<?xml version="1.0"?>
<xsl:stylesheet version="3.0" xmlns:xsl="http://www.w3.org/1999/XSL/Transform">
    <xsl:template match="/">
        <html><body>
            <h1>Book List</h1>
            <xsl:for-each select="//book">
                <p><xsl:value-of select="title"/> - $<xsl:value-of select="price"/></p>
            </xsl:for-each>
        </body></html>
    </xsl:template>
</xsl:stylesheet>"""
executable = xslt_proc.compile_stylesheet(stylesheet_text=xslt)
html = executable.transform_to_string(xdm_node=doc)
print("XSLT produced:", html)
```

---

## Quick Reference

| Task | Method |
|------|--------|
| Load XML from file | `proc.parse_xml(xml_file_name="file.xml")` |
| Load XML from string | `proc.parse_xml(xml_text=xml_string)` |
| Run XPath | `xpath_proc.evaluate("//element")` |
| Run XQuery from string | `xquery_proc.run_query_to_string(query_text=query)` |
| Run XQuery from file | `xquery_proc.run_query_to_string(query_file_name="file.xq")` |
| Run XSLT from string | `xslt_proc.compile_stylesheet(stylesheet_text=xslt)` |
| Run XSLT from file | `xslt_proc.compile_stylesheet(stylesheet_file="file.xsl")` |
| Transform to string | `executable.transform_to_string(xdm_node=doc)` |
| Transform to file | `executable.transform_to_file(xdm_node=doc, output_file="out.xml")` |
</document>